In [4]:
# ==============================================================================
# CUADERNO 04: ITERACIONES CON TRANSFER LEARNING (RESNET-50 Y DENSENET-121)
# ==============================================================================
from google.colab import drive
import os
import shutil
import json
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import f1_score, recall_score, classification_report

# 1. Montar Drive y cargar datos procesados
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/BaseDatos'
PROCESSED_CSV_PATH = os.path.join(BASE_DIR, 'df_procesado_folds.csv')
PREPROC_DIR = os.path.join(BASE_DIR, 'preprocessed_images')
MODELOS_DIR = os.path.join(BASE_DIR, 'modelos_entrenados')
os.makedirs(MODELOS_DIR, exist_ok=True)

assert os.path.exists(PROCESSED_CSV_PATH), "Ejecuta primero el Cuaderno 02."
df_clean = pd.read_csv(PROCESSED_CSV_PATH)

# 2. Copiar imágenes al disco local verificando integridad completa
LOCAL_IMG_DIR = '/content/imagenes_locales'
if os.path.exists(LOCAL_IMG_DIR) and len(os.listdir(LOCAL_IMG_DIR)) < len(df_clean):
    print("Detectada carpeta local incompleta. Limpiando para volver a copiar...")
    shutil.rmtree(LOCAL_IMG_DIR)

if not os.path.exists(LOCAL_IMG_DIR):
    print("Copiando imágenes al almacenamiento local...")
    shutil.copytree(PREPROC_DIR, LOCAL_IMG_DIR)

archivos_locales = set(os.listdir(LOCAL_IMG_DIR))
df_clean = df_clean[df_clean['filename'].isin(archivos_locales)].reset_index(drop=True)
print(f"✓ Imágenes verificadas en disco local: {len(df_clean)}")

clases_ordenadas = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo activo: {device}")

# 3. DataLoaders
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class ODIRDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.img_dir, row['filename'])).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(row['label_idx'], dtype=torch.long)

train_df = df_clean[df_clean['fold'] != 0]
val_df = df_clean[df_clean['fold'] == 0]
train_loader = DataLoader(ODIRDataset(train_df, LOCAL_IMG_DIR, train_transforms), batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(ODIRDataset(val_df, LOCAL_IMG_DIR, val_transforms), batch_size=32, shuffle=False, num_workers=2)

conteos = train_df['label_idx'].value_counts().sort_index().values
pesos_clases = (1.0 / torch.tensor(conteos, dtype=torch.float32))
pesos_clases = (pesos_clases / pesos_clases.sum() * len(clases_ordenadas)).to(device)
criterion = nn.CrossEntropyLoss(weight=pesos_clases)

# 4. Función de entrenamiento para Transfer Learning
def ejecutar_iteracion_tl(arquitectura, epochs=10):
    if arquitectura == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, 8)
    elif arquitectura == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        model.classifier = nn.Linear(model.classifier.in_features, 8)
    model = model.to(device)

    optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    mejor_f1 = 0.0
    historial = {'train_loss': [], 'val_loss': [], 'val_f1_macro': [], 'val_recall_macro': []}
    print(f"\n🚀 Iniciando Iteración Transfer Learning: {arquitectura.upper()}")

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
        epoch_train_loss = running_loss / len(train_loader.dataset)

        model.eval()
        val_loss, preds_all, labels_all = 0.0, [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                val_loss += criterion(outputs, labels).item() * inputs.size(0)
                preds_all.extend(torch.max(outputs, 1)[1].cpu().numpy())
                labels_all.extend(labels.cpu().numpy())

        epoch_val_loss = val_loss / len(val_loader.dataset)
        val_f1 = f1_score(labels_all, preds_all, average='macro', zero_division=0)
        val_rec = recall_score(labels_all, preds_all, average='macro', zero_division=0)
        scheduler.step(val_f1)

        historial['train_loss'].append(epoch_train_loss)
        historial['val_loss'].append(epoch_val_loss)
        historial['val_f1_macro'].append(val_f1)
        historial['val_recall_macro'].append(val_rec)

        print(f"Época {epoch+1:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val F1-Macro: {val_f1:.4f}")

        if val_f1 > mejor_f1:
            mejor_f1 = val_f1
            mejores_preds, mejores_labels = preds_all, labels_all
            torch.save(model.state_dict(), os.path.join(MODELOS_DIR, f'mejor_{arquitectura}_tl.pth'))

    with open(os.path.join(MODELOS_DIR, f'hist_{arquitectura}_tl.json'), 'w') as f:
        json.dump(historial, f)

    print(f"\n✅ Mejor F1-Macro ({arquitectura.upper()}): {mejor_f1:.4f}")
    print(classification_report(mejores_labels, mejores_preds, target_names=clases_ordenadas, zero_division=0))

# Ejecutar ambas iteraciones
ejecutar_iteracion_tl("resnet50", epochs=10)
ejecutar_iteracion_tl("densenet121", epochs=10)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Detectada carpeta local incompleta. Limpiando para volver a copiar...
Copiando imágenes al almacenamiento local...
✓ Imágenes verificadas en disco local: 6392
Dispositivo activo: cuda

🚀 Iniciando Iteración Transfer Learning: RESNET50
Época 01/10 | Train Loss: 1.4667 | Val Loss: 1.7216 | Val F1-Macro: 0.3732
Época 02/10 | Train Loss: 1.2122 | Val Loss: 1.2067 | Val F1-Macro: 0.4700
Época 03/10 | Train Loss: 1.0537 | Val Loss: 1.2579 | Val F1-Macro: 0.4825
Época 04/10 | Train Loss: 0.9783 | Val Loss: 1.1661 | Val F1-Macro: 0.5107
Época 05/10 | Train Loss: 0.9193 | Val Loss: 1.2749 | Val F1-Macro: 0.5224
Época 06/10 | Train Loss: 0.8301 | Val Loss: 1.1728 | Val F1-Macro: 0.4961
Época 07/10 | Train Loss: 0.8080 | Val Loss: 1.2283 | Val F1-Macro: 0.5021
Época 08/10 | Train Loss: 0.7545 | Val Loss: 1.2014 | Val F1-Macro: 0.5049
Época 09/10 | Train Loss: 0.5865 | V

100%|██████████| 30.8M/30.8M [00:00<00:00, 122MB/s]



🚀 Iniciando Iteración Transfer Learning: DENSENET121
Época 01/10 | Train Loss: 1.4820 | Val Loss: 1.4486 | Val F1-Macro: 0.3778
Época 02/10 | Train Loss: 1.2070 | Val Loss: 1.2481 | Val F1-Macro: 0.4542
Época 03/10 | Train Loss: 1.1168 | Val Loss: 1.2730 | Val F1-Macro: 0.4577
Época 04/10 | Train Loss: 1.0335 | Val Loss: 1.5226 | Val F1-Macro: 0.4714
Época 05/10 | Train Loss: 0.9430 | Val Loss: 1.3029 | Val F1-Macro: 0.4645
Época 06/10 | Train Loss: 0.9165 | Val Loss: 1.0830 | Val F1-Macro: 0.5713
Época 07/10 | Train Loss: 0.9079 | Val Loss: 1.3224 | Val F1-Macro: 0.4515
Época 08/10 | Train Loss: 0.8117 | Val Loss: 1.1987 | Val F1-Macro: 0.4980
Época 09/10 | Train Loss: 0.7931 | Val Loss: 1.2586 | Val F1-Macro: 0.5091
Época 10/10 | Train Loss: 0.6703 | Val Loss: 1.1237 | Val F1-Macro: 0.5308

✅ Mejor F1-Macro (DENSENET121): 0.5713
              precision    recall  f1-score   support

           N       0.69      0.73      0.71       571
           D       0.69      0.50      0.58    